# 08 - 带测试模式的批量 Map 分析

这一节把单块分析升级为可缓存、可重试的批量框架，并针对不同章节使用不同结构：

- `Business`：只提取有引用的业务事实，不在 Map 阶段生成分析推断。
- `Risk Factors`：强制区分潜在风险、已发生事项和混合状态。
- `MD&A`：拆分期间、数值、单位、变化方向和管理层解释，并校验数字。

当前进入受控批次模式：按章节筛选，并限制每轮最多新增的 API 调用数。成功结果会单独缓存，重新运行时不会重复调用。

## 1. 读取 chunks 并配置测试模式

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field, field_validator

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data/sec/AAPL"
CACHE_DIR = DATA_DIR / "map_analysis_v3"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
TEST_CHUNK_IDS = [
    "AAPL_risk_factors_001",
    "AAPL_mda_002",
]
SECTION_FILTER = "business"
MAX_NEW_CALLS = 3
PROMPT_VERSIONS = {
    "business": "filing-map-business-v3.1",
    "risk_factors": "filing-map-risk-v3",
    "mda": "filing-map-mda-v3.1",
}
TEMPERATURE = 0
MAX_ATTEMPTS = 2
REQUEST_DELAY_SECONDS = 1.0

chunk_files = sorted(DATA_DIR.glob("*_chunks.json"), reverse=True)
if not chunk_files:
    raise FileNotFoundError("没有找到 chunks JSON，请先运行 05_chunk_sections.ipynb")

chunks_path = chunk_files[0]
chunks_data = json.loads(chunks_path.read_text(encoding="utf-8"))
all_chunks = chunks_data["chunks"]
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in all_chunks}

print(f"总文本块：{len(all_chunks)}")
print(f"测试模式：{TEST_MODE}")
print(f"章节筛选：{SECTION_FILTER}")
print(f"本轮最多新增调用：{MAX_NEW_CALLS}")
print(f"缓存目录：{CACHE_DIR}")

## 2. 创建连续证据块

每个分析 chunk 再切成不超过 500 字符的证据块。它们比 HTML 原始行完整，同时仍保留在整份财报中的精确位置。

In [ ]:
evidence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator="end",
    add_start_index=True,
)


def build_evidence_units(chunk: dict) -> list[dict]:
    documents = evidence_splitter.create_documents([chunk["text"]])
    units = []

    for index, document in enumerate(documents, start=1):
        local_start = document.metadata["start_index"]
        local_end = local_start + len(document.page_content)
        assert chunk["text"][local_start:local_end] == document.page_content

        units.append({
            "evidence_id": f"E{index:03d}",
            "text": document.page_content,
            "display_text": re.sub(r"\s+", " ", document.page_content),
            "local_start": local_start,
            "local_end": local_end,
            "source_start": chunk["source_start"] + local_start,
            "source_end": chunk["source_start"] + local_end,
        })

    return units


def format_numbered_evidence(units: list[dict]) -> str:
    return "\n".join(
        f"[{unit['evidence_id']}] {unit['display_text']}" for unit in units
    )

## 3. 定义三类标准输出

Pydantic 前置校验会统一模型偶尔返回的中文枚举、数字编号和单值列表。

In [ ]:
Materiality = Literal["high", "medium", "low"]
BusinessCategory = Literal[
    "business_model", "product", "service", "geography",
    "market", "distribution", "competition", "supply_chain",
    "research_development", "intellectual_property", "customer", "other",
]
RiskStatus = Literal["potential", "realized", "mixed"]
Direction = Literal["increase", "decrease", "flat", "not_stated"]
MetricUnit = Literal["usd_millions", "usd_billions", "percent", "other"]


def as_list(value):
    if value is None:
        return []
    values = value if isinstance(value, list) else [value]
    return [item for item in values if str(item).strip()]


def normalize_id(value: str | int, prefix: str) -> str:
    text = str(value).strip().upper()
    numbers = re.findall(r"\d+", text)
    if numbers:
        return f"{prefix}{int(numbers[-1]):03d}"
    if text.startswith(prefix):
        text = text[len(prefix):]
    return f"{prefix}{text}"


def normalize_level(value: str) -> str:
    return {"高": "high", "中": "medium", "低": "low"}.get(
        str(value).strip(), str(value).strip()
    )


class BusinessFact(BaseModel):
    fact_id: str
    category: BusinessCategory
    fact_cn: str
    evidence_ids: list[str]
    materiality: Materiality

    @field_validator("fact_id", mode="before")
    @classmethod
    def normalize_fact_id(cls, value):
        return normalize_id(value, "F")

    @field_validator("category", mode="before")
    @classmethod
    def normalize_business_category(cls, value):
        key = re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")
        aliases = {
            "products": "product",
            "services": "service",
            "markets": "market",
            "supply_chain": "supply_chain",
            "research_and_development": "research_development",
            "intellectual_property": "intellectual_property",
        }
        return aliases.get(key, key if key in BusinessCategory.__args__ else "other")

    @field_validator("evidence_ids", mode="before")
    @classmethod
    def normalize_evidence_ids(cls, value):
        return [normalize_id(item, "E") for item in as_list(value)]

    @field_validator("materiality", mode="before")
    @classmethod
    def normalize_materiality(cls, value):
        return normalize_level(value)


class BusinessAnalysis(BaseModel):
    chunk_id: str = ""
    summary_cn: str = ""
    facts: list[BusinessFact]
    chunk_gaps_cn: list[str] = Field(default_factory=list)

    @field_validator("chunk_gaps_cn", mode="before")
    @classmethod
    def normalize_gaps(cls, value):
        return as_list(value)


class RiskItem(BaseModel):
    risk_id: str
    category: str
    risk_cn: str
    status: RiskStatus
    causes_cn: list[str]
    potential_impacts_cn: list[str]
    evidence_ids: list[str]
    materiality: Materiality

    @field_validator("risk_id", mode="before")
    @classmethod
    def normalize_risk_id(cls, value):
        return normalize_id(value, "R")

    @field_validator("status", mode="before")
    @classmethod
    def normalize_status(cls, value):
        aliases = {
            "潜在": "potential",
            "可能": "potential",
            "已发生": "realized",
            "现实": "realized",
            "混合": "mixed",
        }
        text = str(value).strip()
        return aliases.get(text, text)

    @field_validator("causes_cn", "potential_impacts_cn", mode="before")
    @classmethod
    def normalize_text_lists(cls, value):
        return as_list(value)

    @field_validator("evidence_ids", mode="before")
    @classmethod
    def normalize_risk_evidence(cls, value):
        return [normalize_id(item, "E") for item in as_list(value)]

    @field_validator("materiality", mode="before")
    @classmethod
    def normalize_risk_materiality(cls, value):
        return normalize_level(value)


class RiskAnalysis(BaseModel):
    chunk_id: str = ""
    summary_cn: str = ""
    risks: list[RiskItem]
    chunk_gaps_cn: list[str] = Field(default_factory=list)

    @field_validator("chunk_gaps_cn", mode="before")
    @classmethod
    def normalize_risk_gaps(cls, value):
        return as_list(value)


class FinancialMetric(BaseModel):
    metric_id: str
    metric_name: str
    current_period: str
    current_value: str
    unit: MetricUnit
    comparison_period: str | None = None
    comparison_value: str | None = None
    change_value: str | None = None
    change_direction: Direction
    management_explanation_cn: str | None = None
    evidence_ids: list[str]
    materiality: Materiality

    @field_validator(
        "current_period", "current_value", "comparison_period",
        "comparison_value", "change_value", mode="before"
    )
    @classmethod
    def normalize_numeric_fields(cls, value):
        if value is None:
            return None
        text = str(value).strip()
        if text.lower() in {"not_stated", "not stated", "none", "null", "未提供"}:
            return None
        return text

    @field_validator("management_explanation_cn", mode="before")
    @classmethod
    def normalize_explanation(cls, value):
        if value is None or str(value).strip() in {"", "未提供", "not_stated"}:
            return None
        return str(value).strip()

    @field_validator("metric_id", mode="before")
    @classmethod
    def normalize_metric_id(cls, value):
        return normalize_id(value, "M")

    @field_validator("unit", mode="before")
    @classmethod
    def normalize_unit(cls, value):
        aliases = {
            "百万美元": "usd_millions",
            "美元百万": "usd_millions",
            "十亿美元": "usd_billions",
            "百分比": "percent",
            "%": "percent",
            "其他": "other",
        }
        text = str(value).strip()
        return aliases.get(text, text)

    @field_validator("change_direction", mode="before")
    @classmethod
    def normalize_direction(cls, value):
        aliases = {
            "增加": "increase",
            "上升": "increase",
            "减少": "decrease",
            "下降": "decrease",
            "持平": "flat",
            "未说明": "not_stated",
        }
        text = str(value).strip()
        return aliases.get(text, text)

    @field_validator("evidence_ids", mode="before")
    @classmethod
    def normalize_metric_evidence(cls, value):
        return [normalize_id(item, "E") for item in as_list(value)]

    @field_validator("materiality", mode="before")
    @classmethod
    def normalize_metric_materiality(cls, value):
        return normalize_level(value)


class MDAAnalysis(BaseModel):
    chunk_id: str = ""
    summary_cn: str = ""
    metrics: list[FinancialMetric]
    chunk_gaps_cn: list[str] = Field(default_factory=list)

    @field_validator("chunk_gaps_cn", mode="before")
    @classmethod
    def normalize_mda_gaps(cls, value):
        return as_list(value)

## 4. 为不同章节定义 Prompt

Map 阶段只提取直接披露的信息，不生成需要外部知识的投资判断。

In [ ]:
COMMON_SYSTEM = """你是一名审慎的 SEC 财报信息提取助手。
只能依据输入证据分析，不使用外部知识，不提供投资建议。
只能引用真实存在的 E 编号，例如 E003；不得自己编写引文。
所有编号必须保留字母前缀并作为 JSON 字符串返回。
high、medium、low 等枚举值必须使用指定英文，不得翻译。
chunk_gaps_cn 只表示当前文本块缺少的信息，不代表整份财报没有披露。
只返回一个 JSON 对象。"""

BUSINESS_SYSTEM = COMMON_SYSTEM + """
提取 4 到 8 条直接披露的业务事实，不生成分析推断。
每条事实包含 fact_id、category、fact_cn、evidence_ids、materiality。
category 只能使用 business_model、product、service、geography、market、distribution、competition、supply_chain、research_development、intellectual_property、customer、other。"""

RISK_SYSTEM = COMMON_SYSTEM + """
提取 2 到 6 项风险。每项包含 risk_id、category、risk_cn、status、causes_cn、potential_impacts_cn、evidence_ids、materiality。
status 只能是 potential、realized、mixed。
仅有 may、could、can、might、risk、uncertainty 等条件性表述时，必须标记 potential，不得写成已经发生。
只有原文明示已经发生或正在发生时才能使用 realized；同时包含现实情况和未来可能影响时使用 mixed。"""

MDA_SYSTEM = COMMON_SYSTEM + """
提取 6 到 12 项重要财务指标。每项包含 metric_id、metric_name、current_period、current_value、unit、comparison_period、comparison_value、change_value、change_direction、management_explanation_cn、evidence_ids、materiality。
不要机械列出全部地区或全部产品。若原文同时包含 Segment Operating Performance、Products and Services Performance、Gross Margin，结果必须覆盖这三个子章节，并至少包含一个地区指标、一个产品或服务指标、一个毛利额指标和一个毛利率指标。
unit 只能是 usd_millions、usd_billions、percent、other。change_direction 只能是 increase、decrease、flat、not_stated。
数值保持原文格式；不得自行计算原文未披露的数值。若记录美元指标，证据必须同时覆盖 dollars in millions 等单位说明和对应数字。
management_explanation_cn 只能翻译管理层明确披露的变化原因，不能猜测。"""


def make_prompt(system_prompt: str) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        (
            "human",
            """chunk_id: {chunk_id}
ticker: {ticker}
form: {form}
section: {section_title}

<numbered_evidence>
{numbered_evidence}
</numbered_evidence>""",
        ),
    ])


ANALYSIS_CONFIG = {
    "business": (BusinessAnalysis, make_prompt(BUSINESS_SYSTEM)),
    "risk_factors": (RiskAnalysis, make_prompt(RISK_SYSTEM)),
    "mda": (MDAAnalysis, make_prompt(MDA_SYSTEM)),
}

## 5. 创建模型、校验器和缓存分析函数

In [ ]:
load_dotenv(PROJECT_ROOT / ".env", override=True)
api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
if not api_key:
    raise ValueError("请先在 .env 中配置 DEEPSEEK_API_KEY")

model_name = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
llm = ChatDeepSeek(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
    temperature=TEMPERATURE,
    request_timeout=60,
    max_retries=1,
)


def normalize_numeric_text(value: str) -> str:
    text = str(value).replace(",", "").replace("$", "").replace("%", "")
    text = re.sub(r"\(([-+]?\d+(?:\.\d+)?)\)", r"-\1", text)
    return re.sub(r"\s+", "", text)


def parse_number(value: str | None) -> float | None:
    if value is None:
        return None
    normalized = normalize_numeric_text(value)
    match = re.search(r"[-+]?\d+(?:\.\d+)?", normalized)
    return float(match.group()) if match else None


def resolve_evidence(ids: list[str], unit_map: dict[str, dict], errors: list[str], owner_id: str):
    resolved = []
    for evidence_id in ids:
        unit = unit_map.get(evidence_id)
        if unit is None:
            errors.append(f"{owner_id} 使用了无效证据编号 {evidence_id}")
        else:
            resolved.append(unit)
    return resolved


def validate_business(analysis: BusinessAnalysis, unit_map: dict[str, dict]):
    errors = []
    resolved = {}
    if not 4 <= len(analysis.facts) <= 8:
        errors.append("Business facts 数量不在 4 到 8 之间")
    for fact in analysis.facts:
        resolved[fact.fact_id] = resolve_evidence(
            fact.evidence_ids, unit_map, errors, fact.fact_id
        )
    return errors, resolved


def validate_risk(analysis: RiskAnalysis, unit_map: dict[str, dict]):
    errors = []
    resolved = {}
    modality_cues = (
        "may", "could", "can", "might", "risk", "uncertainty",
        "subject to", "exposed to", "depend", "potential",
    )
    if not 2 <= len(analysis.risks) <= 6:
        errors.append("Risk items 数量不在 2 到 6 之间")

    for risk in analysis.risks:
        evidence = resolve_evidence(risk.evidence_ids, unit_map, errors, risk.risk_id)
        resolved[risk.risk_id] = evidence
        evidence_text = " ".join(unit["display_text"] for unit in evidence).lower()
        if risk.status == "potential" and not any(cue in evidence_text for cue in modality_cues):
            errors.append(f"{risk.risk_id} 标记为 potential，但证据中没有条件性语气")

    return errors, resolved


def validate_mda(analysis: MDAAnalysis, unit_map: dict[str, dict]):
    errors = []
    resolved = {}
    if not 6 <= len(analysis.metrics) <= 12:
        errors.append("MD&A metrics 数量不在 6 到 12 之间")

    for metric in analysis.metrics:
        evidence = resolve_evidence(
            metric.evidence_ids, unit_map, errors, metric.metric_id
        )
        resolved[metric.metric_id] = evidence
        evidence_text = " ".join(unit["display_text"] for unit in evidence)
        normalized_evidence = normalize_numeric_text(evidence_text)

        numeric_fields = {
            "current_value": metric.current_value,
            "comparison_value": metric.comparison_value,
            "change_value": metric.change_value,
        }
        for field_name, value in numeric_fields.items():
            if value and normalize_numeric_text(value) not in normalized_evidence:
                errors.append(
                    f"{metric.metric_id} 的 {field_name}={value} 无法在证据中找到"
                )

        for period_name, period in (
            ("current_period", metric.current_period),
            ("comparison_period", metric.comparison_period),
        ):
            if period and str(period) not in evidence_text:
                errors.append(
                    f"{metric.metric_id} 的 {period_name}={period} 无法在证据中找到"
                )

        if metric.unit == "usd_millions" and "dollars in millions" not in evidence_text.lower():
            errors.append(f"{metric.metric_id} 缺少 dollars in millions 单位证据")

        current_number = parse_number(metric.current_value)
        comparison_number = parse_number(metric.comparison_value)
        if current_number is not None and comparison_number is not None:
            expected_direction = (
                "increase" if current_number > comparison_number
                else "decrease" if current_number < comparison_number
                else "flat"
            )
            if metric.change_direction != expected_direction:
                errors.append(
                    f"{metric.metric_id} 方向应为 {expected_direction}，"
                    f"模型返回 {metric.change_direction}"
                )

    all_evidence_text = " ".join(
        unit["display_text"] for unit in unit_map.values()
    ).lower()
    metric_names = [metric.metric_name.lower() for metric in analysis.metrics]
    if "segment operating performance" in all_evidence_text:
        regional_terms = ("americas", "europe", "greater china", "japan", "asia pacific")
        if not any(any(term in name for term in regional_terms) for name in metric_names):
            errors.append("MD&A 缺少地区分部指标")
    if "products and services performance" in all_evidence_text:
        product_terms = ("iphone", "mac", "ipad", "wearables", "services net sales")
        if not any(any(term in name for term in product_terms) for name in metric_names):
            errors.append("MD&A 缺少产品或服务指标")
    if "gross margin" in all_evidence_text:
        if not any("gross margin" in name and "percentage" not in name for name in metric_names):
            errors.append("MD&A 缺少毛利额指标")
        if not any("gross margin percentage" in name for name in metric_names):
            errors.append("MD&A 缺少毛利率指标")

    return errors, resolved


VALIDATORS = {
    "business": validate_business,
    "risk_factors": validate_risk,
    "mda": validate_mda,
}


def parse_model_output(raw_content, section: str, chunk_id: str, schema):
    if not isinstance(raw_content, str):
        raw_content = str(raw_content)
    raw_content = raw_content.strip()
    if raw_content.startswith("```"):
        raw_content = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_content)
    payload = json.loads(raw_content)

    output_key = {
        "business": "facts",
        "risk_factors": "risks",
        "mda": "metrics",
    }[section]
    if isinstance(payload, list):
        payload = {output_key: payload}
    if not isinstance(payload, dict):
        raise ValueError("模型 JSON 顶层必须是对象或数组")

    aliases = {
        "business_facts": "facts",
        "risk_items": "risks",
        "financial_metrics": "metrics",
    }
    for alias, standard_name in aliases.items():
        if alias in payload and standard_name not in payload:
            payload[standard_name] = payload.pop(alias)

    payload.setdefault("chunk_id", chunk_id)
    payload.setdefault("summary_cn", "")
    payload.setdefault("chunk_gaps_cn", [])
    return schema.model_validate(payload)


def load_current_cache(chunk: dict) -> dict | None:
    cache_path = CACHE_DIR / f"{chunk['chunk_id']}.json"
    if not cache_path.exists():
        return None
    cached = json.loads(cache_path.read_text(encoding="utf-8"))
    cache_version = cached.get("run_metadata", {}).get("prompt_version")
    expected_version = PROMPT_VERSIONS[chunk["section"]]
    if (
        cached.get("validation", {}).get("structure_passed")
        and cache_version == expected_version
    ):
        return cached
    return None


def analyze_chunk(chunk: dict, force: bool = False) -> dict:
    cache_path = CACHE_DIR / f"{chunk['chunk_id']}.json"
    cached = None if force else load_current_cache(chunk)
    if cached is not None:
        print(f"缓存命中：{chunk['chunk_id']}")
        return cached

    schema, prompt = ANALYSIS_CONFIG[chunk["section"]]
    units = build_evidence_units(chunk)
    unit_map = {unit["evidence_id"]: unit for unit in units}
    numbered_evidence = format_numbered_evidence(units)
    json_llm = llm.bind(response_format={"type": "json_object"})
    chain = prompt | json_llm

    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw_message = chain.invoke({
                "chunk_id": chunk["chunk_id"],
                "ticker": chunk["ticker"],
                "form": chunk["form"],
                "section_title": chunk["section_title"],
                "numbered_evidence": numbered_evidence,
            })
            analysis = parse_model_output(
                raw_message.content, chunk["section"], chunk["chunk_id"], schema
            )
            analysis.chunk_id = analysis.chunk_id or chunk["chunk_id"]
            if not analysis.summary_cn:
                if isinstance(analysis, RiskAnalysis):
                    analysis.summary_cn = "；".join(
                        item.risk_cn for item in analysis.risks[:3]
                    )
                elif isinstance(analysis, MDAAnalysis):
                    analysis.summary_cn = "；".join(
                        f"{item.metric_name}: {item.current_value}"
                        for item in analysis.metrics[:4]
                    )
                else:
                    analysis.summary_cn = "；".join(
                        item.fact_cn for item in analysis.facts[:3]
                    )
            if isinstance(analysis, BusinessAnalysis):
                for index, item in enumerate(analysis.facts, start=1):
                    item.fact_id = f"F{index:03d}"
            elif isinstance(analysis, RiskAnalysis):
                for index, item in enumerate(analysis.risks, start=1):
                    item.risk_id = f"R{index:03d}"
            else:
                for index, item in enumerate(analysis.metrics, start=1):
                    item.metric_id = f"M{index:03d}"

            errors, resolved = VALIDATORS[chunk["section"]](analysis, unit_map)
            output = {
                "run_metadata": {
                    "prompt_version": PROMPT_VERSIONS[chunk["section"]],
                    "analyzed_at_utc": datetime.now(timezone.utc).isoformat(),
                    "model": model_name,
                    "temperature": TEMPERATURE,
                    "attempt": attempt,
                    "token_usage": raw_message.usage_metadata or {},
                },
                "chunk_metadata": {
                    key: chunk[key]
                    for key in (
                        "chunk_id", "ticker", "form", "section",
                        "source_start", "source_end",
                    )
                },
                "model_output": analysis.model_dump(),
                "resolved_evidence": resolved,
                "validation": {
                    "structure_passed": not errors,
                    "structure_errors": errors,
                    "semantic_review_status": "pending",
                },
            }
            cache_path.write_text(
                json.dumps(output, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
            return output
        except Exception as exc:
            last_error = exc
            print(f"{chunk['chunk_id']} 第 {attempt} 次失败：{exc}")
            if attempt < MAX_ATTEMPTS:
                time.sleep(2)

    raise RuntimeError(f"{chunk['chunk_id']} 分析失败：{last_error}")

## 6. 选择测试块或受控批次

测试模式使用固定样本；批次模式先按章节过滤，再包含所有有效缓存和最多 `MAX_NEW_CALLS` 个待处理块。

In [ ]:
if TEST_MODE:
    chunks_to_analyze = [chunks_by_id[chunk_id] for chunk_id in TEST_CHUNK_IDS]
else:
    candidates = [
        chunk for chunk in all_chunks
        if SECTION_FILTER is None or chunk["section"] == SECTION_FILTER
    ]
    cached_candidates = [
        chunk for chunk in candidates if load_current_cache(chunk) is not None
    ]
    pending_candidates = [
        chunk for chunk in candidates if load_current_cache(chunk) is None
    ]
    chunks_to_analyze = [
        *cached_candidates,
        *pending_candidates[:MAX_NEW_CALLS],
    ]

print(f"本次计划处理 {len(chunks_to_analyze)} 个文本块：")
for chunk in chunks_to_analyze:
    state = "cached" if load_current_cache(chunk) is not None else "new"
    print(f"- {chunk['chunk_id']} ({chunk['section']}, {state})")

## 7. 执行测试 Map

这个单元格会调用 DeepSeek。已经通过校验并缓存的 chunk 会直接读取缓存。

In [ ]:
map_results = []
batch_stats = {
    "cache_hits": 0,
    "new_successes": 0,
    "validation_failed": 0,
    "exceptions": [],
}

for index, chunk in enumerate(chunks_to_analyze, start=1):
    print(f"\n[{index}/{len(chunks_to_analyze)}] {chunk['chunk_id']}")
    was_cached = load_current_cache(chunk) is not None
    try:
        result = analyze_chunk(chunk)
        map_results.append(result)
        validation = result["validation"]
        print(f"结构校验：{validation['structure_passed']}")
        for error in validation["structure_errors"]:
            print(f"- {error}")

        if was_cached:
            batch_stats["cache_hits"] += 1
        elif validation["structure_passed"]:
            batch_stats["new_successes"] += 1
        else:
            batch_stats["validation_failed"] += 1
    except Exception as exc:
        error = {"chunk_id": chunk["chunk_id"], "error": str(exc)}
        batch_stats["exceptions"].append(error)
        print(f"批次继续，当前块失败：{exc}")

    if index < len(chunks_to_analyze) and not was_cached:
        time.sleep(REQUEST_DELAY_SECONDS)

batch_status_path = CACHE_DIR / "latest_batch_status.json"
batch_status_path.write_text(
    json.dumps({
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "test_mode": TEST_MODE,
        "section_filter": SECTION_FILTER,
        "max_new_calls": MAX_NEW_CALLS,
        "stats": batch_stats,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

## 8. 查看测试结果摘要

In [ ]:
print(f"批次统计：{batch_stats}\n")

for result in map_results:
    metadata = result["chunk_metadata"]
    output = result["model_output"]
    validation = result["validation"]
    item_count = len(
        output.get("facts")
        or output.get("risks")
        or output.get("metrics")
        or []
    )
    print("=" * 80)
    print(f"chunk：{metadata['chunk_id']}")
    print(f"类型：{metadata['section']}")
    print(f"条目数：{item_count}")
    print(f"结构校验：{validation['structure_passed']}")
    print(f"摘要：{output['summary_cn']}")
    print(f"Token：{result['run_metadata']['token_usage']}")

## 下一步

当前先完成 Business 的小批次。检查结果后，继续运行即可处理剩余 Business 缓存；随后将 `SECTION_FILTER` 切换为 `risk_factors` 和 `mda`，逐批完成全量 Map。